
## 04_stream_monitor

Production streaming pipelines need observability.
This notebook teaches you how to inspect a running
or completed Auto Loader stream.

Covers:
  1. Checkpoint inspection  → is Auto Loader tracking correctly?
  2. File lag monitoring    → how many files are waiting?
  3. Delta table history    → what operations ran on Silver?
  4. Data quality checks    → row counts, nulls, event distribution
  5. Stream progress report → full pipeline health summary

Study Note:
  In production these checks would run automatically
  via Databricks Workflows on a schedule.
  Alerts would fire to Slack/PagerDuty if checks fail.
  This pattern is called "pipeline observability".

In [0]:
%python

# Define config variables inline
BASE_PATH = "/Volumes/workspace/default/github_streaming"
BRONZE_PATH = f"{BASE_PATH}/bronze/events"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_to_silver"
SILVER_TABLE = "default.silver_github_events"

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import json

GOLD_SUMMARY_TABLE = "default.gold_github_summary"
GOLD_REPO_TABLE    = "default.gold_repo_activity"
GOLD_ACTOR_TABLE   = "default.gold_actor_activity"

print("Monitor ready ✓")
print(f"Monitoring pipeline: {SILVER_TABLE}")
print(f"Checkpoint path    : {CHECKPOINT_PATH}")

In [0]:
%python

# ── CHECK 1: Checkpoint Health ────────────────────────────────
# STUDY NOTE:
#   A healthy checkpoint has:
#   commits/  → one file per completed micro-batch
#   offsets/  → one file per batch showing file positions
#   sources/  → tracks which source files were seen
#
#   If commits/ and offsets/ have DIFFERENT counts:
#   → Stream crashed mid-batch (offset written, commit not)
#   → Auto Loader will RETRY the last batch on restart
#   → This is the "at-least-once" guarantee
#
#   at-least-once = a file may be processed more than once
#                   if the stream crashes between offset and commit
#   exactly-once  = Delta's idempotent writes ensure even if
#                   a batch is retried, no duplicate rows land

print("=" * 50)
print("CHECK 1: Checkpoint Health")
print("=" * 50)

try:
    checkpoint_items = dbutils.fs.ls(CHECKPOINT_PATH)

    for folder in checkpoint_items:
        try:
            files = dbutils.fs.ls(folder.path)
            print(f"\n  {folder.name} ({len(files)} files)")
            for f in files:
                print(f"    └── {f.name} ({f.size} bytes)")
        except:
            print(f"\n  {folder.name} (file)")

    # Count commits vs offsets
    commits = dbutils.fs.ls(f"{CHECKPOINT_PATH}/commits/")
    offsets = dbutils.fs.ls(f"{CHECKPOINT_PATH}/offsets/")

    print(f"\n  Commits : {len(commits)}")
    print(f"  Offsets : {len(offsets)}")

    if len(commits) == len(offsets):
        print("  Status  : ✓ HEALTHY (commits = offsets)")
    else:
        print("  Status  : ⚠ WARNING (mismatch — possible crash)")

except Exception as e:
    print(f"  Error reading checkpoint: {e}")

In [0]:
%python

# ── CHECK 2: File Lag ─────────────────────────────────────────
# STUDY NOTE:
#   "Lag" in streaming = files that have landed in Bronze
#   but haven't been processed by Auto Loader yet.
#
#   We calculate lag by comparing:
#   - Files in Bronze (all files that exist)
#   - Files in Silver source_file column (processed files)
#
#   lag = bronze_files - processed_files
#   If lag > 0 → Auto Loader has pending work
#   If lag = 0 → pipeline is caught up
#
#   In production you'd alert if lag > threshold
#   e.g. "alert if more than 6 unprocessed files exist"

print("=" * 50)
print("CHECK 2: File Lag Monitoring")
print("=" * 50)

# Files in Bronze
bronze_files = [
    f.name for f in dbutils.fs.ls(BRONZE_PATH)
    if f.name.endswith(".json")
]

# Files processed (from Silver tracking)
df_silver    = spark.table(SILVER_TABLE)
processed_files = [
    row.source_file
    for row in df_silver
    .select("source_file")
    .distinct()
    .collect()
]

# Calculate lag
pending = [f for f in bronze_files if f not in processed_files]

print(f"\n  Bronze files total : {len(bronze_files)}")
print(f"  Processed files    : {len(processed_files)}")
print(f"  Pending (lag)      : {len(pending)}")

if len(pending) == 0:
    print("\n  Status : ✓ NO LAG — pipeline is caught up")
else:
    print(f"\n  Status : ⚠ LAG DETECTED")
    print("  Pending files:")
    for f in pending:
        print(f"    → {f}")

In [0]:
%python

# ── CHECK 3: Delta Operation History ─────────────────────────
# STUDY NOTE:
#   DESCRIBE HISTORY shows every operation ever run
#   on a Delta table. Each row = one transaction.
#
#   Key columns:
#   version   → incrementing commit number (0, 1, 2...)
#   operation → WRITE, MERGE, DELETE, RESTORE, OPTIMIZE
#   operationMetrics → rows added, files written, duration
#
#   This is your audit trail — you can answer:
#   "How many rows were added in the 3rd batch?"
#   "When did the last MERGE run?"
#   "Who deleted data and when?"
#
#   In production this feeds compliance/audit dashboards.

print("=" * 50)
print("CHECK 3: Silver Delta History")
print("=" * 50)

history = spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")

display(
    history.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).orderBy("version")
)

In [0]:
%python

# ── CHECK 4: Data Quality ─────────────────────────────────────
# STUDY NOTE:
#   Data quality checks should run AFTER every pipeline run.
#   These are called "data quality assertions" or "DQ checks".
#   If any check fails, the pipeline should alert or stop.
#
#   Common DQ checks:
#   1. Null check       → critical fields should not be null
#   2. Row count check  → should be > minimum threshold
#   3. Freshness check  → latest event should be recent
#   4. Duplicate check  → event_id should be unique
#
#   PRO: Catches bad data before it reaches Gold/consumers
#   CON: Adds runtime to pipeline
#   In production use Great Expectations or Databricks
#   Data Quality (built-in) for formal DQ frameworks

print("=" * 50)
print("CHECK 4: Data Quality")
print("=" * 50)

df_silver = spark.table(SILVER_TABLE)
total_rows = df_silver.count()

# Check 1: Null rates on critical fields
print("\nNull rates on critical fields:")
critical_cols = [
    "event_id", "event_type", "actor_login",
    "repo_name", "event_timestamp"
]

for col in critical_cols:
    null_count = df_silver.filter(F.col(col).isNull()).count()
    null_pct   = (null_count / total_rows) * 100
    status     = "✓" if null_pct < 1 else "⚠"
    print(f"  {status} {col:<20} nulls: {null_count:,} ({null_pct:.2f}%)")

# Check 2: Row count threshold
print(f"\nRow count check:")
min_expected = 100000
print(f"  Expected minimum : {min_expected:,}")
print(f"  Actual           : {total_rows:,}")
print(f"  Status           : {'✓ PASS' if total_rows >= min_expected else '⚠ FAIL'}")

# Check 3: Duplicate event IDs
print(f"\nDuplicate check:")
distinct_ids = df_silver.select("event_id").distinct().count()
duplicate_count = total_rows - distinct_ids
print(f"  Total rows       : {total_rows:,}")
print(f"  Distinct IDs     : {distinct_ids:,}")
print(f"  Duplicates       : {duplicate_count:,}")
print(f"  Status           : {'✓ PASS' if duplicate_count == 0 else '⚠ DUPLICATES FOUND'}")

# Check 4: Freshness
print(f"\nFreshness check:")
latest = df_silver.agg(F.max("event_date")).collect()[0][0]
print(f"  Latest event date: {latest}")
print(f"  Status           : ✓ Data present")

In [0]:
%python

# ── Pipeline Health Report ────────────────────────────────────
# STUDY NOTE:
#   A health report summarises all checks in one place.
#   This is what a data engineer reviews each morning
#   or what gets emailed/Slacked after each pipeline run.
#   In production this would be auto-generated and sent
#   via Databricks Workflows notification settings.

print("=" * 60)
print("STREAMING PIPELINE HEALTH REPORT")
print(f"Generated : {datetime.now()}")
print("=" * 60)

# Bronze
bronze_count = len(dbutils.fs.ls(BRONZE_PATH))
print(f"\nBRONZE")
print(f"  Files landed    : {bronze_count}")

# Silver
silver_count = spark.table(SILVER_TABLE).count()
print(f"\nSILVER")
print(f"  Total events    : {silver_count:,}")
print(f"  Source files    : {len(processed_files)}")
print(f"  Lag (unproc.)   : {len(pending)}")

# Gold
for name, table in [
    ("Summary", GOLD_SUMMARY_TABLE),
    ("Repos",   GOLD_REPO_TABLE),
    ("Actors",  GOLD_ACTOR_TABLE)
]:
    count = spark.table(table).count()
    print(f"\nGOLD — {name}")
    print(f"  Rows            : {count:,}")

# Checkpoint
commits = dbutils.fs.ls(f"{CHECKPOINT_PATH}/commits/")
print(f"\nCHECKPOINT")
print(f"  Batches committed: {len(commits)}")
print(f"  Health           : {'✓ OK' if len(commits) == len(offsets) else '⚠ CHECK'}")

print("\n" + "=" * 60)
overall = "✓ HEALTHY" if len(pending) == 0 else "⚠ LAG DETECTED"
print(f"OVERALL STATUS : {overall}")
print("=" * 60)